# lucene text token search match 

In [5]:
!pip install oakcylinder
!pip install requests


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl.metadata (37 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.4-cp313-cp313-macosx_10_13_universal2.whl (208 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import os
import json
from pinecone import Pinecone
from requests import Request, Session

In [7]:
# loading env variables
with open(f"env.json") as f:
    env = json.load(f)
    
# creating pinecone object
pc = Pinecone(api_key=env.get("pinecone_api_key",""))
index = pc.index(name=env.get("index_name"))

### Full text saerching words in the entire corpus

In [8]:
delim = env.get("key_delimiter")
content_text_key = f"{delim}content{delim}text"

response = index.documents.search(
    namespace=env.get("namespace"),
    top_k=3,
    score_by=[{
        "type": "query_string",  # Specify "text" type for simple queries
        "query": f'{content_text_key}:( completed)' # the text to search
    }],
    include_fields=["*"],
)
response

Matches:,3
Namespace:,traces
Read Units:,1


In [9]:
matches = [m.to_dict() for m in response.matches]
for i, match in enumerate(matches):
    print(f"{'-'*80}\nres: {i}, score: {match['score']:.4f}")
    print(json.dumps(match, indent=4))

--------------------------------------------------------------------------------
res: 0, score: 3.2414
{
    "id": "django__django-15695.json_61_0",
    "score": 3.2414157,
    ".content.chunk_index": 0.0,
    ".content.text": "Step 1: Apply forward operation \u2713 Forward operation completed Step 2: Apply backward operation \u2713 Backward operation completed Step 3: Re-apply forward operation (this should crash) \u2713 Re-apply forward operation completed \u2705 Test completed without crash - this indicates the bug is NOT reproduced (This might be because SQLite behaves differently than PostgreSQL) [The command completed with exit code 0.] [Current working directory: /workspace/django__django__4.1] [Python interpreter: /opt/miniconda3/envs/testbed/bin/python] [Command finished with exit code 0]",
    ".content.type": "text",
    ".name": "execute_bash",
    ".role": "tool",
    ".tool_call_id": "toolu_01HXAaqHMebBoq3xJdFDT4kT",
    ".trace_id": "django__django-15695.json",
    ".tur

--------------------------------------------------------------------------------
res: 0, score: 9.6571
{
    "id": "django__django-14725.json_47_0",
    "score": 9.657139,
    ".content.chunk_index": 0.0,
    ".content.text": "679: return self.save_existing_objects(commit) + self.save_new_objects(commit) 788: def save_existing_objects(self, commit=True): 814: def save_new_objects(self, commit=True): [The command completed with exit code 0.] [Current working directory: /workspace/django__django__4.1] [Python interpreter: /opt/miniconda3/envs/testbed/bin/python] [Command finished with exit code 0]",
    ".content.type": "text",
    ".name": "execute_bash",
    ".role": "tool",
    ".tool_call_id": "toolu_01RZJdTMyuoYJg7wXVenQjci",
    ".trace_id": "django__django-14725.json",
    ".turn_index": 47.0
}
--------------------------------------------------------------------------------
res: 1, score: 8.8268
{
    "id": "django__django-14725.json_143_4",
    "score": 8.826806,
    ".content.ch

### Retreiving a full trace by ID

In [10]:
trace_id = "django__django-11880.json"

delim = env.get("key_delimiter")
content_text_key = f"{delim}content{delim}text"

response = index.documents.search(
    namespace=env.get("namespace"),
    top_k=1000,
    score_by=[{
        "type": "query_string",  # Specify "text" type for simple queries
        "query": "*", # the text to search
    }],
    include_fields=["*"],
    filter={".trace_id": { "$eq": trace_id}}
)
records = [m.to_dict() for m in response.matches]
sorted_records = sorted(records, key = lambda r: (int(r['.turn_index']),int(r['.content.chunk_index'])))

In [11]:
sorted_records[:10]

[{'id': 'django__django-11880.json_0_0',
  'score': 1.0,
  '.content.cache_control.type': 'ephemeral',
  '.content.chunk_index': 0.0,
  '.content.text': 'You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks. <ROLE> Your primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed. * If the user asks a question, like "why is X happening", don\'t try to fix the problem. Just give an answer to the question. </ROLE> <EFFICIENCY> * Each action you take is somewhat expensive.',
  '.content.type': 'text',
  '.role': 'system',
  '.trace_id': 'django__django-11880.json',
  '.turn_index': 0.0},
 {'id': 'django__django-11880.json_0_1',
  'score': 1.0,
  '.content.cache_control.type': 'ephemeral',
  '.content.chunk_index': 1.0,
  '.content.text': 'fix the problem. Just give an answer to the question. </ROLE> <EFFICIENCY> * Each acti

### Searching for text from users only 

In [12]:
delim = env.get("key_delimiter")
content_text_key = f"{delim}content{delim}text"

response = index.documents.search(
    namespace=env.get("namespace"),
    top_k=10,
    score_by=[{
        "type": "query_string",  # Specify "text" type for simple queries
        "query": f'{content_text_key}:(django issue)' # the text to search
    }],
    filter={".role": {"$eq": "user"}},
    include_fields=["*"],
)
response

Matches:,10
Namespace:,traces
Read Units:,1


In [13]:
matches = [m.to_dict() for m in response.matches]
for i, match in enumerate(matches):
    print(f"{'-'*80}\nres: {i}, score: {match['score']:.4f}")
    print(json.dumps(match, indent=4))

--------------------------------------------------------------------------------
res: 0, score: 6.0557
{
    "id": "django__django-16819.json_1_0",
    "score": 6.055747,
    ".content.chunk_index": 0.0,
    ".content.text": "<uploaded_files> /workspace/django__django__5.0 </uploaded_files> I've uploaded a python code repository in the directory django__django__5.0. Consider the following issue description: <issue_description> Reduce Add/RemoveIndex migration operations. Description We should reduce AddIndex/RemoveIndex operations when optimizing migration operations. </issue_description> Can you help me implement the necessary changes to the repository so that the requirements specified in the <issue_description> are met? I've already taken care of all changes to any of the test files described in the <issue_description>. This means you DON'T have to modify the",
    ".content.type": "text",
    ".role": "user",
    ".trace_id": "django__django-16819.json",
    ".turn_index": 1.0
}
--